# 26 — Cross-package holdout evaluation v2

Codex coder agent가 작성·실행하는 별도 v2 연구입니다. 기존 v1 holdout 실행이 아니며, v1 준수 결과를 주장하지 않습니다. 같은 sealed data를 참조하되 OLD-D20-K3와 STRUCT-D20-K3 end-to-end package를 비교합니다.

이 노트북은 누수 방지를 위해 (1) candidate/execution contract seal, (2) query-only embedding preflight, (3) label-free ranking/bundle freeze, (4) label projection/evaluation 순서를 강제합니다. 이번 실행은 query embedding이 없으면 2단계에서 정상적으로 멈춥니다.

## 결과 전 고정 계약

- 공통 검색: Vector Top50 + normalized BM25 Top50, Vector:BM25=0.4:0.6, BM25 k1=1.5/b=0.75, RRF k=60, component depth=50, fused chunk depth=20, 최종 unique-card K=3.
- OLD: 327개 기존 chunk 중 section/benefit만 사용하고, query-text regex로 proper/numeric은 RRF 유지, semantic만 BGE rerank합니다.
- STRUCT: 147개 direct-body structural chunk, frozen same-card deterministic 1-hop bundle(max 5 chunks), all-query BGE rerank입니다.
- BGE: local-only revision 953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e, trust_remote_code=False, max_length=8192, only_second, fp16, batch=2. raw single logit만 사용하며 tie는 원 RRF rank 다음 card/chunk ID입니다.
- ranking과 bundle이 label보다 먼저 freeze되어야 합니다. label을 연 뒤 ranking/bundle 코드 재실행·수정은 금지합니다.
- span normalization은 NFKC + casefold + whitespace collapse입니다. 인위적으로 붙인 heading/card-name 문자열은 span 충족 근거가 아니며 실제 source evidence/body만 인정합니다.
- Card Hit@3 27/30, Supported-card Hit@3 14/20, Evidence Span Coverage@3 macro 0.65가 absolute gate입니다. Supported-card metric은 v1 strict와 동등하지 않은 정책 연속값입니다.
- invalid면 invalid; 한 후보만 absolute gate 통과 시 그 후보; 둘 다 실패면 holdout_failed. 둘 다 통과하고 Supported Hit 차이가 2/20 이상이면 Card Hit·Coverage·Supported MRR 비회귀일 때만 우승합니다. Supported Hit 동률이면 Coverage +0.025 이상, Supported MRR·Card Hit 비회귀, paired wins>losses일 때만 우승합니다. 그 외 inconclusive입니다. 결과 후 gate 변경이나 저점 재실행은 금지합니다.

In [1]:
from pathlib import Path
import csv, hashlib, importlib.metadata as md, json, os, platform, sys, time

cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError(f'Unexpected cwd: {cwd}')
NOTEBOOK=ROOT/'notebooks/26_cross_package_holdout_evaluation.ipynb'
OUT=ROOT/'notebooks/data/26_cross_package_holdout_evaluation'; OUT.mkdir(parents=True,exist_ok=True)
H14=ROOT/'notebooks/data/14_retrieval_holdout'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def write_json(name,obj): (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def cell_sha(cell_id):
    doc=json.loads(NOTEBOOK.read_text(encoding='utf-8'))
    cell=next(c for c in doc['cells'] if c.get('id')==cell_id)
    src=''.join(cell.get('source',[]))
    return hashlib.sha256(canon({'cell_id':cell_id,'source_text':src}).encode()).hexdigest()

# HOLDOUT QUERY/LABEL FILES ARE INTENTIONALLY NOT OPENED OR HASHED IN THIS CELL.
holdout_safe={
 'holdout_contract.json':H14/'holdout_contract.json',
 'holdout_data_manifest.json':H14/'holdout_data_manifest.json',
 'holdout_generation_audit.json':H14/'holdout_generation_audit.json',
}
holdout_safe_hashes={k:sha(v) for k,v in holdout_safe.items()}
h_contract=json.loads(holdout_safe['holdout_contract.json'].read_text())
h_manifest=json.loads(holdout_safe['holdout_data_manifest.json'].read_text())
if h_manifest['status']!='locked_not_evaluated' or h_manifest['candidate_config_seal']['status']!='pending':
    raise RuntimeError('v1 holdout is not in locked_not_evaluated/pending state')
for rel,digest in h_manifest['file_sha256'].items():
    if rel.endswith('holdout_queries.json') or rel.endswith('holdout_labels.json'): continue
    p=ROOT/rel
    if sha(p)!=digest: raise RuntimeError(f'Holdout safe source changed: {rel}')

source_rel=[
 'notebooks/13_hierarchical_chunking_retrieval.ipynb',
 'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl',
 'notebooks/data/13_hierarchical_chunking_retrieval/embedding_usage.json',
 'notebooks/data/13_hierarchical_chunking_retrieval/input_manifest.json',
 'notebooks/data/13_hierarchical_chunking_retrieval/index_manifest.json',
 'notebooks/21_current_chunking_prebranch_reranker_evaluation.ipynb',
 'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/evaluation_contract.json',
 'notebooks/data/21_current_chunking_prebranch_reranker_evaluation/phase_a_classification.csv',
 'notebooks/22_structural_heading_chunking_ablation.ipynb',
 'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl',
 'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl',
 'notebooks/data/22_structural_heading_chunking_ablation/embedding_usage.json',
 'notebooks/data/22_structural_heading_chunking_ablation/followup2_contract.json',
 'notebooks/24_multicard_recommendation_retrieval_evaluation.ipynb',
 'notebooks/data/24_multicard_recommendation_retrieval_evaluation/followup1_contract.json',
 'notebooks/25_llm_answer_quality_pipeline_comparison.ipynb',
 'notebooks/data/25_llm_answer_quality_pipeline_comparison/evaluation_contract.json',
 'notebooks/data/25_llm_answer_quality_pipeline_comparison/ranking_freeze.jsonl',
 'notebooks/data/25_llm_answer_quality_pipeline_comparison/structural_direct_bundles.jsonl',
 'notebooks/data/25_llm_answer_quality_pipeline_comparison/local_bge_scorer_audit.csv',
 'notebooks/data/25_llm_answer_quality_pipeline_comparison/local_bge_resources.json',
]
model_rel=[
 '.cache/reranker/bge-reranker-v2-m3/config.json',
 '.cache/reranker/bge-reranker-v2-m3/tokenizer_config.json',
 '.cache/reranker/bge-reranker-v2-m3/tokenizer.json',
 '.cache/reranker/bge-reranker-v2-m3/sentencepiece.bpe.model',
 '.cache/reranker/bge-reranker-v2-m3/model.safetensors',
]
for rel in source_rel+model_rel:
    if not (ROOT/rel).is_file(): raise RuntimeError(f'Missing frozen source: {rel}')
source_hashes={rel:sha(ROOT/rel) for rel in source_rel+model_rel}
source_cell_hashes={}
for rel,ids in {
 'notebooks/21_current_chunking_prebranch_reranker_evaluation.ipynb':['phase-a-classifier','phase-b-candidates-inputs'],
 'notebooks/24_multicard_recommendation_retrieval_evaluation.ipynb':['24-followup1-build'],
 'notebooks/25_llm_answer_quality_pipeline_comparison.ipynb':['25-build-rankings-bundles','25-local-bge-score'],
}.items():
    doc=json.loads((ROOT/rel).read_text())
    for cid in ids:
        c=next(x for x in doc['cells'] if x.get('id')==cid); src=''.join(c.get('source',[]))
        source_cell_hashes[f'{rel}::{cid}']=hashlib.sha256(canon({'cell_id':cid,'source_text':src}).encode()).hexdigest()

contract={
 'schema_version':'cross_package_holdout_v2_preflight_1.0',
 'provenance':'Codex coder agent',
 'study_identity':{'v1_execution':False,'v1_files_mutated':False,'relationship':'separate cross-package retrieval/evidence study referencing the same sealed data; not a v1-compliant result'},
 'sequence':['candidate_execution_seal_v2','query_text_only_cache_lookup','label_free_ranking_and_bundle_freeze','labels_projection_and_evaluation'],
 'common_retrieval':{'vector_top_n':50,'normalized_bm25_top_n':50,'vector_weight':0.4,'bm25_weight':0.6,'bm25_k1':1.5,'bm25_b':0.75,'rrf_k':60,'component_depth':50,'fused_chunk_depth':20,'output_unique_card_k':3,'raw_score_mixing':False,'tie_break':'score desc -> component best rank -> chunk_id lexical'},
 'packages':{
  'OLD-D20-K3':{'corpus':'327 card/page/section/benefit chunks','eligible_levels':['section','benefit'],'route':'query-text regex: proper_noun/numeric_condition -> RRF passthrough; semantic -> BGE','reranker_input':'safe title augmentation from source fields only','evidence_packaging':'25 frozen source artifact contract'},
  'STRUCT-D20-K3':{'corpus':'147 structural direct-body searchable chunks','route':'all-query BGE','bundle':'deterministic same-card 1-hop; max 5 unique chunks; root-children fanout/recursive/cross-card/whole-doc forbidden','evidence_packaging':'25 frozen source artifact contract'},
 },
 'bge':{'revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','architecture':'XLMRobertaForSequenceClassification','trust_remote_code':False,'local_files_only':True,'max_length':8192,'truncation':'only_second','dtype':'float16','batch_size':2,'score':'raw single logit desc','tie':'original RRF rank then card/chunk ID','score_mixing':False},
 'normalization':{'span':'NFKC + casefold + whitespace collapse','actual_body_only':True,'augmentation_cannot_satisfy_span':True,'duplicate_rule':'identical normalized spans count once per query','nested_rule':'a normalized span wholly contained in a longer frozen span counts as the longer span only when both refer to the same source claim; otherwise each frozen span remains a separate unit'},
 'denominators':{'all':30,'evidence':20,'numeric':10,'semantic':10},
 'absolute_gates':{'card_hit_at_3':{'minimum_count':27,'denominator':30},'supported_card_hit_at_3':{'minimum_count':14,'denominator':20},'evidence_span_coverage_at_3_macro':{'minimum':0.65}},
 'metrics':{'supported_card_hit_at_3':'expected card in Top3 and at least one frozen normalized span is in the actual final payload body/evidence','policy_continuity_note':'not equivalent to v1 strict','coverage':'macro unique normalized frozen-span coverage in Top3 payloads','other':['card_mrr_at_3','supported_card_mrr_at_3','numeric10','semantic10','paired_win_loss_tie','complete_coverage_count']},
 'validity':['candidate and execution seals exact','fixed denominators exact','label-free rankings and bundles frozen before labels','every frozen evidence span projects into both full corpora','no metric N/A'],
 'winner_rule':{'invalid':'invalid','one_absolute_pass':'that package','neither_absolute_pass':'holdout_failed','both_pass_supported_hit_gap_at_least_2_of_20':'winner only if Card Hit, Coverage, Supported MRR non-regressive','both_pass_supported_hit_tie':'winner only if Coverage delta >= 0.025, Supported MRR and Card Hit non-regressive, paired wins > losses','otherwise':'inconclusive'},
 'execution':{'environment':'skn25','api_key_read':False,'external_api_requests':0,'network_requests':0,'new_embeddings':0,'gpu':0,'chroma_queries':0,'package_installs':0,'low_score_rerun_forbidden':True},
 'reporting':{'holdout_question_or_answer_text_in_summary':False},
}
write_json('evaluation_contract.json',contract)
package_names=['numpy','pandas','scipy','scikit-learn','torch','transformers','tokenizers','huggingface-hub','safetensors','sentencepiece','einops','tiktoken','nbclient','nbformat']
packages={n:(md.version(n) if n in {d.metadata['Name'] for d in md.distributions() if d.metadata.get('Name')} else None) for n in package_names}
seal_core={
 'schema_version':'candidate_execution_seal_v2_1.0',
 'created_before_holdout_query_or_label_content_read':True,
 'v1_status_observed':h_manifest['status'],
 'v1_candidate_seal_observed':h_manifest['candidate_config_seal']['status'],
 'v1_unexecuted_and_immutable':True,
 'v2_not_a_v1_compliant_result':True,
 'dataset_id':h_contract['dataset_id'],
 'holdout_safe_source_hashes':holdout_safe_hashes,
 'manifest_declared_query_sha256_not_recomputed':h_manifest['file_sha256']['notebooks/data/14_retrieval_holdout/holdout_queries.json'],
 'manifest_declared_label_sha256_not_recomputed':h_manifest['file_sha256']['notebooks/data/14_retrieval_holdout/holdout_labels.json'],
 'contract_sha256':sha(OUT/'evaluation_contract.json'),
 'source_hashes':source_hashes,
 'source_code_cell_hashes':source_cell_hashes,
 'execution_code_cell_hashes':{'candidate_seal':cell_sha('26-candidate-seal'),'query_preflight':cell_sha('26-query-preflight')},
 'packages':packages,
 'candidate_config_status':'sealed',
 'label_free_execution_status':'not_started_waiting_for_query_embedding_preflight',
}
seal={'seal_core':seal_core,'seal_core_sha256':hashlib.sha256(canon(seal_core).encode()).hexdigest()}
write_json('candidate_execution_seal_v2.json',seal)
write_json('source_manifest_before.json',{'files':source_hashes,'holdout_safe_files':holdout_safe_hashes,'query_and_label_raw_hashes_read':False})
print({'candidate_seal':seal['seal_core_sha256'],'query_content_read':False,'label_content_read':False,'status':'sealed'})

{'candidate_seal': 'a9f5547d372e4af6f21b749e1b70dc59fd42f37d7570ba3d502b3dcbfb40653c', 'query_content_read': False, 'label_content_read': False, 'status': 'sealed'}


In [2]:
# This cell may read query text only after the candidate seal is validated. It never opens labels.
import numpy as np, unicodedata
seal=json.loads((OUT/'candidate_execution_seal_v2.json').read_text())
if hashlib.sha256(canon(seal['seal_core']).encode()).hexdigest()!=seal['seal_core_sha256']: raise RuntimeError('candidate seal mismatch')
if seal['seal_core']['candidate_config_status']!='sealed': raise RuntimeError('candidate config not sealed')
if seal['seal_core']['execution_code_cell_hashes']['query_preflight']!=cell_sha('26-query-preflight'): raise RuntimeError('query preflight code changed after seal')
queries_path=H14/'holdout_queries.json'
query_raw_sha=sha(queries_path)
if query_raw_sha!=seal['seal_core']['manifest_declared_query_sha256_not_recomputed']: raise RuntimeError('sealed holdout query file mismatch')
qdoc=json.loads(queries_path.read_text(encoding='utf-8'))
qrows=qdoc if isinstance(qdoc,list) else qdoc.get('queries')
if not isinstance(qrows,list) or len(qrows)!=30: raise RuntimeError('expected 30 sealed queries')
def qtext(r):
    for key in ('query','query_text','text','question'):
        if isinstance(r.get(key),str) and r[key].strip(): return r[key]
    raise RuntimeError('query text field missing')
def qid(r):
    for key in ('query_id','id'):
        if isinstance(r.get(key),str) and r[key].strip(): return r[key]
    raise RuntimeError('query id field missing')
ordered=[{'query_id':qid(r),'text':qtext(r)} for r in qrows]
if len({r['query_id'] for r in ordered})!=30: raise RuntimeError('query IDs are not unique')
for r in ordered: r['text_sha256']=hashlib.sha256(r['text'].encode()).hexdigest()
try:
    import tiktoken
    enc=tiktoken.get_encoding('cl100k_base')
except Exception as exc: raise RuntimeError(f'offline tiktoken unavailable: {type(exc).__name__}')
for r in ordered: r['token_count']=len(enc.encode(r['text']))

cache_files=[]
for usage_rel in ['notebooks/data/13_hierarchical_chunking_retrieval/embedding_usage.json','notebooks/data/22_structural_heading_chunking_ablation/embedding_usage.json']:
    usage=json.loads((ROOT/usage_rel).read_text())
    if 'batches' in usage:
        for b in usage['batches']:
            if b.get('cache_path'): cache_files.append(ROOT/b['cache_path'])
            elif b.get('batch_fingerprint'): cache_files.append((ROOT/usage_rel).parent/'embedding_cache/text-embedding-3-small'/f"{b['batch_fingerprint']}.npz")
for rel in ['notebooks/data/24_multicard_recommendation_retrieval_evaluation/embedding_cache/text-embedding-3-small/031928bee3ba51aa700afbbb6903e0a1e839a1afcfbe7a4e45dbdb5a8d4b2027.npz','notebooks/data/24_multicard_recommendation_retrieval_evaluation/embedding_cache/text-embedding-3-small/fca33efe64c4dfe0f728c9c7e7eeb7d5ea9b61a48b571f75416bca1f2bc9454b.npz']:
    p=ROOT/rel
    if p.is_file(): cache_files.append(p)
cache_files=sorted(set(cache_files),key=lambda p:p.as_posix())
available={}
cache_audit=[]
for p in cache_files:
    with np.load(p,allow_pickle=False) as z:
        hs=[str(x) for x in z['hashes']]; vec=z['embeddings']
        if vec.dtype!=np.float32 or vec.ndim!=2 or vec.shape[1]!=1536 or not np.isfinite(vec).all(): raise RuntimeError(f'bad cache: {p}')
        for i,h in enumerate(hs): available.setdefault(h,[]).append({'path':p.relative_to(ROOT).as_posix(),'row':i})
        cache_audit.append({'path':p.relative_to(ROOT).as_posix(),'sha256':sha(p),'items':len(hs),'dimension':1536,'dtype':'float32','finite':True})
hits=[r for r in ordered if r['text_sha256'] in available]
missing=[r for r in ordered if r['text_sha256'] not in available]
approval_core={
 'purpose':'query-only embeddings for sealed cross-package holdout v2',
 'model':'text-embedding-3-small',
 'ordered_queries':missing,
 'item_cap':len(missing),
 'token_cap':sum(r['token_count'] for r in missing),
 'max_requests':1 if missing else 0,
 'transmitted_allowlist':['ordered query text only'],
 'transmitted_forbidden':['documents','chunks','labels','expected card','expected level','relevant IDs','evidence spans','required terms','category','metrics','scores','metadata','secrets'],
 'query_file_sha256':query_raw_sha,
 'candidate_seal_sha256':seal['seal_core_sha256'],
}
approval={'approval_core':approval_core,'approval_core_sha256':hashlib.sha256(canon(approval_core).encode()).hexdigest(),'status':'approval_required' if missing else 'not_required'}
write_json('query_embedding_approval_manifest.json',approval)
preflight={
 'status':'awaiting_query_embedding_approval' if missing else 'all_query_embeddings_cached_ready_for_label_free_execution',
 'query_count':30,'query_file_sha256':query_raw_sha,'cached_query_count':len(hits),'missing_query_count':len(missing),
 'missing_query_tokens':sum(r['token_count'] for r in missing),'cache_files':cache_audit,
 'labels_opened_or_hashed':False,'api_key_read':False,'external_api_requests':0,'network_requests':0,'new_embeddings':0,'gpu':0,'model_load':0,'chroma_queries':0,
 'ranking_bundle_started':False,'label_free_execution_seal_created':False,
}
write_json('query_embedding_preflight.json',preflight)
write_json('execution_status.json',{'candidate_seal':'sealed','query_preflight':preflight['status'],'label_free_execution':'not_started','labels':'not_opened','evaluation':'not_started'})
readme=f"""# 26 cross-package holdout v2

이 디렉터리는 기존 v1 실행이 아니라, 같은 sealed data를 참조하는 별도 OLD/STRUCT end-to-end retrieval/evidence 연구의 사전 동결 자료입니다. v1 파일과 상태는 수정하지 않았습니다.

현재 상태: `{preflight['status']}`. candidate config는 query/label을 열기 전에 seal했습니다. 이후 query text만 열어 로컬 cache exact lookup을 수행했으며 label은 열거나 해시하지 않았습니다.

- 로컬 cache hit: {len(hits)}/30
- 신규 query embedding 필요: {len(missing)}개, cl100k 예상 {sum(r['token_count'] for r in missing)} tokens, 최대 {1 if missing else 0} request
- 현재 API/network/new embedding/GPU/model/Chroma: 모두 0

질의 또는 정답 평문은 README에 기록하지 않습니다. 승인이 필요한 경우 `query_embedding_approval_manifest.json`의 immutable core와 SHA를 사용해야 하며, 문서·gold·평가 필드는 전송 금지입니다. query embedding이 준비되기 전에는 ranking/bundle/label/evaluation 단계로 진행하지 않습니다.

지표는 Card Hit@3, supported-card Hit@3, actual payload body 기준 Evidence Span Coverage@3를 중심으로 합니다. supported-card는 기존 v1 strict와 동등한 지표가 아닙니다.
"""
(OUT/'README.md').write_text(readme,encoding='utf-8')
# Allowed inputs must remain unchanged; holdout labels remain unread.
for rel,digest in seal['seal_core']['source_hashes'].items():
    if sha(ROOT/rel)!=digest: raise RuntimeError(f'source changed during preflight: {rel}')
for name,digest in seal['seal_core']['holdout_safe_source_hashes'].items():
    if sha(H14/name)!=digest: raise RuntimeError(f'holdout safe source changed: {name}')
print({'status':preflight['status'],'cached':len(hits),'missing':len(missing),'tokens':preflight['missing_query_tokens'],'approval_sha256':approval['approval_core_sha256'],'labels_read':False})

{'status': 'awaiting_query_embedding_approval', 'cached': 0, 'missing': 30, 'tokens': 1202, 'approval_sha256': '879d97790713321c673635a4b36d4935794623b31b6e0797f48528554f860229', 'labels_read': False}


In [3]:
# Offline preflight finalization only; no ranking/model/label work is permitted in this run.
preflight=json.loads((OUT/'query_embedding_preflight.json').read_text())
if preflight['missing_query_count']==0: raise RuntimeError('All embeddings are cached; implement the separately sealed label-free execution stage before labels')
required=['evaluation_contract.json','candidate_execution_seal_v2.json','source_manifest_before.json','query_embedding_approval_manifest.json','query_embedding_preflight.json','execution_status.json','README.md']
for name in required:
    if not (OUT/name).is_file(): raise RuntimeError(f'missing output: {name}')
manifest_files={name:sha(OUT/name) for name in required}
integrity={
 'status':'PASS_PREFLIGHT_AWAITING_APPROVAL',
 'notebook_code_cells_compile':True,
 'candidate_seal_before_query_read':True,
 'query_count':30,'query_embeddings_cached':preflight['cached_query_count'],'query_embeddings_missing':preflight['missing_query_count'],
 'labels_opened_or_hashed':False,'ranking_bundle_started':False,
 'current_run':{'api':0,'network':0,'new_embedding':0,'gpu':0,'model_load':0,'chroma':0,'package_install':0},
 'source_before_after_exact':True,'v1_modified':False,'output_hashes':manifest_files,
}
write_json('integrity.json',integrity)
run_files=required+['integrity.json']
write_json('run_manifest.json',{'manifest_self_hash_excluded':True,'files':{name:sha(OUT/name) for name in run_files},'notebook_source_cell_hashes':{cid:cell_sha(cid) for cid in ['26-candidate-seal','26-query-preflight','26-offline-finalize']},'execution':'skn25 fresh kernel CPU/offline preflight','labels_read':False})
print({'integrity':'PASS_PREFLIGHT_AWAITING_APPROVAL','outputs':len(run_files)+1,'external_requests':0})

{'integrity': 'PASS_PREFLIGHT_AWAITING_APPROVAL', 'outputs': 9, 'external_requests': 0}


## 승인 후 실행 단계

아래 세 셀은 기본 실행에서 제외됩니다. 첫 셀은 승인된 query text 30개만 한 번 embedding하고, 둘째 셀은 label을 열지 않은 채 검색·bundle·BGE 결과를 봉인하며, 셋째 셀만 봉인을 검증한 뒤 label을 열어 평가합니다. 각 셀은 현재 source SHA를 별도 환경변수와 대조하므로 승인 뒤 코드가 바뀌면 OpenAI client, GPU model, label load 전에 중단됩니다.

In [1]:
# Approved query-only embedding. Do not execute without the exact approval and cell-source hashes.
from pathlib import Path
import hashlib,json,os,time
import numpy as np
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None
if ROOT is None: raise RuntimeError(f'Unexpected cwd: {cwd}')
NOTEBOOK=ROOT/'notebooks/26_cross_package_holdout_evaluation.ipynb'; OUT=ROOT/'notebooks/data/26_cross_package_holdout_evaluation'; H14=ROOT/'notebooks/data/14_retrieval_holdout'
APPROVAL_SHA='879d97790713321c673635a4b36d4935794623b31b6e0797f48528554f860229'; MODEL='text-embedding-3-small'
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def write_json(name,obj): (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def require(ok,msg):
    if not ok: raise RuntimeError(msg)
def cell_sha(cid):
    doc=json.loads(NOTEBOOK.read_text()); c=next((x for x in doc['cells'] if x.get('id')==cid),None); require(c is not None,f'missing cell {cid}'); src=''.join(c.get('source',[])); return hashlib.sha256(canon({'cell_id':cid,'source_text':src}).encode()).hexdigest()
require(os.environ.get('RUN_APPROVED_26_QUERY_EMBEDDING')=='1','query embedding not approved')
require(os.environ.get('APPROVED_26_QUERY_EMBEDDING_SHA256')==APPROVAL_SHA,'approval SHA env mismatch')
current_code_sha=cell_sha('26-approved-query-embedding'); require(os.environ.get('APPROVED_26_API_CELL_SHA256')==current_code_sha,'approved API cell source changed')
approval=json.loads((OUT/'query_embedding_approval_manifest.json').read_text()); core=approval['approval_core']; require(hashlib.sha256(canon(core).encode()).hexdigest()==approval['approval_core_sha256']==APPROVAL_SHA,'approval core mismatch')
require(approval['status']=='approval_required' and core['model']==MODEL and core['item_cap']==30 and core['token_cap']==1202 and core['max_requests']==1,'approval caps/status mismatch')
seal=json.loads((OUT/'candidate_execution_seal_v2.json').read_text()); require(hashlib.sha256(canon(seal['seal_core']).encode()).hexdigest()==seal['seal_core_sha256']==core['candidate_seal_sha256'],'candidate seal mismatch')
require(sha(H14/'holdout_queries.json')==core['query_file_sha256'],'query file mismatch')
qdoc=json.loads((H14/'holdout_queries.json').read_text()); rows=qdoc['queries']; ordered=[]
for r in rows:
    text=next((r[k] for k in ('query','query_text','text','question') if isinstance(r.get(k),str) and r[k].strip()),None); qid=r.get('query_id') or r.get('id'); require(text is not None and isinstance(qid,str),'bad query schema'); ordered.append({'query_id':qid,'text':text,'text_sha256':hashlib.sha256(text.encode()).hexdigest()})
require(len(ordered)==30 and [r['query_id'] for r in ordered]==[r['query_id'] for r in core['ordered_queries']],'ordered query IDs mismatch')
require(all(a['text']==b['text'] and a['text_sha256']==b['text_sha256'] for a,b in zip(ordered,core['ordered_queries'])),'ordered query text mismatch')
require(sum(r['token_count'] for r in core['ordered_queries'])==1202,'approved token count mismatch')
cache_dir=OUT/'embedding_cache'/MODEL; cache_dir.mkdir(parents=True,exist_ok=True); cache_path=cache_dir/f'{APPROVAL_SHA}.npz'; ledger_path=OUT/'query_embedding_attempt_ledger.json'
if cache_path.is_file():
    require(ledger_path.is_file(),'cache exists without attempt ledger'); ledger=json.loads(ledger_path.read_text()); require(ledger.get('status')=='completed' and ledger.get('requests_started')==1 and ledger.get('retry_count')==0,'cache ledger invalid')
    with np.load(cache_path,allow_pickle=False) as z: require(z['embeddings'].shape==(30,1536) and z['embeddings'].dtype==np.float32 and np.isfinite(z['embeddings']).all() and list(z['hashes'])==[r['text_sha256'] for r in ordered],'cached embeddings invalid')
    print({'resume':'complete_cache','api_requests_current':0,'cache_sha256':sha(cache_path)})
else:
    if ledger_path.is_file(): raise RuntimeError('prior query embedding attempt exists; retry is forbidden')
    ledger={'approval_core_sha256':APPROVAL_SHA,'api_cell_source_sha256':current_code_sha,'status':'started','requests_started':1,'retry_count':0,'items':30,'approved_tokens':1202,'started_unix':time.time()}
    write_json('query_embedding_attempt_ledger.json',ledger)
    from dotenv import dotenv_values
    key=dotenv_values(ROOT/'.env').get('OPENAI_API_KEY'); require(isinstance(key,str) and bool(key),'OPENAI_API_KEY missing')
    from openai import OpenAI
    client=OpenAI(api_key=key,max_retries=0); started=time.perf_counter()
    try: response=client.embeddings.create(model=MODEL,input=[r['text'] for r in ordered])
    except Exception as exc:
        ledger.update({'status':'failed_no_retry','error_type':type(exc).__name__,'elapsed_seconds':time.perf_counter()-started}); write_json('query_embedding_attempt_ledger.json',ledger); raise
    elapsed=time.perf_counter()-started; matrix=np.asarray([x.embedding for x in response.data],dtype=np.float32)
    require(matrix.shape==(30,1536) and np.isfinite(matrix).all(),'embedding response invalid'); usage_tokens=int(response.usage.total_tokens)
    require(usage_tokens<=1202,'provider usage exceeds approved token cap')
    np.savez_compressed(cache_path,embeddings=matrix,hashes=np.asarray([r['text_sha256'] for r in ordered]),query_ids=np.asarray([r['query_id'] for r in ordered]))
    ledger.update({'status':'completed','elapsed_seconds':elapsed,'provider_input_tokens':usage_tokens,'cache_sha256':sha(cache_path),'completed_unix':time.time()}); write_json('query_embedding_attempt_ledger.json',ledger)
    write_json('query_embedding_usage.json',{'model':MODEL,'approval_core_sha256':APPROVAL_SHA,'requests':1,'retry_count':0,'items':30,'approved_token_cap':1202,'provider_input_tokens':usage_tokens,'elapsed_seconds':elapsed,'pricing_usd_per_million_input_tokens':0.02,'pricing_source_url':'https://openai.com/api/pricing/','actual_cost_usd':usage_tokens*0.02/1_000_000,'cache_path':cache_path.relative_to(ROOT).as_posix(),'cache_sha256':sha(cache_path),'shape':[30,1536],'dtype':'float32','finite':True,'documents_or_gold_transmitted':False})
    print({'api_requests':1,'items':30,'provider_tokens':usage_tokens,'cache_sha256':sha(cache_path)})

{'api_requests': 1, 'items': 30, 'provider_tokens': 1202, 'cache_sha256': '65da46c8207bf4d5b7e88747f6de102bd4354f8f77da33a9ceff326ee10cfdb9'}


In [1]:
# Label-free OLD/STRUCT retrieval, bundle construction, local BGE scoring, and freeze.
from pathlib import Path
from collections import Counter,defaultdict
from decimal import Decimal
import csv,gc,hashlib,json,math,os,re,resource,subprocess,time,unicodedata
import numpy as np
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None
if ROOT is None: raise RuntimeError(f'Unexpected cwd: {cwd}')
NOTEBOOK=ROOT/'notebooks/26_cross_package_holdout_evaluation.ipynb'; OUT=ROOT/'notebooks/data/26_cross_package_holdout_evaluation'; BGE=ROOT/'.cache/reranker/bge-reranker-v2-m3'; APPROVAL_SHA='879d97790713321c673635a4b36d4935794623b31b6e0797f48528554f860229'
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def require(ok,msg):
    if not ok: raise RuntimeError(msg)
def cell_sha(cid):
    d=json.loads(NOTEBOOK.read_text()); c=next((x for x in d['cells'] if x.get('id')==cid),None); require(c is not None,f'missing cell {cid}'); return hashlib.sha256(canon({'cell_id':cid,'source_text':''.join(c.get('source',[]))}).encode()).hexdigest()
def write_json(name,obj): (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def write_jsonl(name,rows): (OUT/name).write_text(''.join(canon(r)+'\n' for r in rows),encoding='utf-8')
def read_jsonl(p): return [json.loads(x) for x in Path(p).read_text().splitlines() if x.strip()]
require(os.environ.get('RUN_APPROVED_26_LABEL_FREE')=='1' and os.environ.get('CUDA_VISIBLE_DEVICES')=='0','label-free GPU execution not approved for physical GPU0')
require(os.environ.get('RUN_APPROVED_26_LABEL_EVAL','0')=='0','label evaluation must remain disabled')
code_sha=cell_sha('26-label-free-retrieval-bge'); require(os.environ.get('APPROVED_26_LABEL_FREE_CODE_SHA256')==code_sha,'label-free code SHA mismatch')
approval=json.loads((OUT/'query_embedding_approval_manifest.json').read_text()); require(approval['approval_core_sha256']==APPROVAL_SHA and hashlib.sha256(canon(approval['approval_core']).encode()).hexdigest()==APPROVAL_SHA,'approval mismatch')
seal=json.loads((OUT/'candidate_execution_seal_v2.json').read_text()); require(hashlib.sha256(canon(seal['seal_core']).encode()).hexdigest()==seal['seal_core_sha256']==approval['approval_core']['candidate_seal_sha256'],'candidate seal mismatch')
for rel,digest in seal['seal_core']['source_hashes'].items(): require(sha(ROOT/rel)==digest,f'source changed: {rel}')
cache=OUT/'embedding_cache/text-embedding-3-small'/f'{APPROVAL_SHA}.npz'; require(cache.is_file(),'approved query embedding cache missing')
with np.load(cache,allow_pickle=False) as z: qmat=z['embeddings'].copy(); qids=[str(x) for x in z['query_ids']]; qhash=[str(x) for x in z['hashes']]
require(qmat.shape==(30,1536) and qmat.dtype==np.float32 and np.isfinite(qmat).all(),'query cache invalid')
ordered=approval['approval_core']['ordered_queries']; require(qids==[r['query_id'] for r in ordered] and qhash==[r['text_sha256'] for r in ordered],'query cache order/hash mismatch'); query={r['query_id']:r['text'] for r in ordered}; qvec={q:v for q,v in zip(qids,qmat)}
old=read_jsonl(ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl'); new=read_jsonl(ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl'); hierarchy=read_jsonl(ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl')
require(len(old)==327 and len(new)==147,'corpus count mismatch'); old_by={r['id']:r for r in old}; new_by={r['chunk_id']:r for r in new}; node_by={r['node_id']:r for r in hierarchy}; children=defaultdict(list)
for n in hierarchy:
    if n['parent_id'] is not None: children[n['parent_id']].append(n)
for xs in children.values(): xs.sort(key=lambda n:((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
def cache_rows(usage_path):
    usage=json.loads(Path(usage_path).read_text()); mats=[]; hashes=[]
    for b in usage['batches']:
        p=ROOT/b['cache_path'] if b.get('cache_path') else Path(usage_path).parent/'embedding_cache/text-embedding-3-small'/f"{b['batch_fingerprint']}.npz"
        with np.load(p,allow_pickle=False) as z: hashes.extend(str(x) for x in z['hashes']); mats.append(z['embeddings'].copy())
    matrix=np.vstack(mats); require(matrix.dtype==np.float32 and matrix.shape[1]==1536 and np.isfinite(matrix).all(),'document cache invalid'); return hashes,matrix
oh,om=cache_rows(ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/embedding_usage.json'); nh,nm=cache_rows(ROOT/'notebooks/data/22_structural_heading_chunking_ablation/embedding_usage.json')
require(oh[:327]==[hashlib.sha256(r['document'].encode()).hexdigest() for r in old] and nh==[hashlib.sha256(r['retrieval_text'].encode()).hexdigest() for r in new],'document hash/cache positional contract mismatch')
ovec={r['id']:om[i] for i,r in enumerate(old)}; nvec={r['chunk_id']:nm[i] for i,r in enumerate(new)}
RAW_TOKEN=re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*',re.IGNORECASE)
def norm(v): return ' '.join(unicodedata.normalize('NFKC',str(v)).lower().split())
def dec(v):
    x=format(Decimal(str(v).replace(',','')).normalize(),'f'); x=x.rstrip('0').rstrip('.') if '.' in x else x; return '0' if x in {'','-0'} else x
def tokens(v):
    text=norm(v); out=list(RAW_TOKEN.findall(text))
    for run in re.findall(r'[가-힣](?:[가-힣 ]{0,38}[가-힣])?',text):
        joined=run.replace(' ','')
        for size in (2,3,4): out.extend(f'ko{size}_'+joined[i:i+size] for i in range(max(0,len(joined)-size+1)))
    consumed=[]
    for m in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원',text): out.append('money_krw_'+dec(Decimal(m.group(1).replace(',',''))*10000+Decimal(m.group(2).replace(',',''))*1000)); consumed.append(m.span())
    for m in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원',text):
        if not any(a<=m.start() and m.end()<=b for a,b in consumed): out.append('money_krw_'+dec(Decimal(m.group(1).replace(',',''))*{'만':10000,'천':1000,None:1}[m.group(2)]))
    for m in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*%',text): out.append('percent_'+dec(m.group(1)))
    for m in re.finditer(r'(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)',text): out.append('period_'+(m.group(1) or 'none')+'_'+dec(m.group(2))+'_'+m.group(3))
    return out
def bm25(q,docs):
    qt=tokens(q); dt={k:tokens(v) for k,v in docs.items()}; df=Counter(t for ts in dt.values() for t in set(ts)); avg=sum(map(len,dt.values()))/len(dt); score={}
    for cid,ts in dt.items():
        f=Counter(ts); s=0.0
        for t in qt:
            if f[t]: s+=math.log(1+(len(dt)-df[t]+.5)/(df[t]+.5))*f[t]*2.5/(f[t]+1.5*(.25+.75*len(ts)/avg))
        score[cid]=s
    return sorted(score,key=lambda c:(-score[c],c))
def vector(qv,vectors):
    s={c:float(np.sum((v-qv)**2,dtype=np.float64)) for c,v in vectors.items()}; require(all(math.isfinite(v) for v in s.values()),'nonfinite vector score'); return sorted(s,key=lambda c:(s[c],c))
def rrf(v,b):
    score=defaultdict(float); vr={c:i for i,c in enumerate(v[:50],1)}; br={c:i for i,c in enumerate(b[:50],1)}
    for c,i in vr.items(): score[c]+=.4/(60+i)
    for c,i in br.items(): score[c]+=.6/(60+i)
    return sorted(score,key=lambda c:(-score[c],min(vr.get(c,10**9),br.get(c,10**9)),c))[:50],score
PROPER=[re.compile(r'(?:어느|어떤)\s*(?:(?:카드사|은행|회사)\s*)?상품(?:인가)?$'),re.compile(r'(?:어느\s*)?(?:카드사|은행|회사)(?:인가)?$'),re.compile(r'(?:발급사|발행사)(?:는|은|가|인가)?$'),re.compile(r'어디서\s*(?:발급|발행|출시)')]
NUMERIC=[re.compile(r'얼마(?:인가|나)?$'),re.compile(r'몇\s*(?:원|%|퍼센트|마일|마일리지|포인트|회|개월|일|년)(?:인가)?$'),re.compile(r'얼마나\s*(?:할인|적립|차감|청구)'),re.compile(r'(?:할인율|적립률|연회비|수수료|(?:할인|적립)\s*(?:금액|한도)|(?:월|연간)\s*(?:할인|적립)?\s*한도|리터당\s*할인\s*금액|(?:마일리지|포인트)\s*적립\s*기준|실적\s*(?:금액|기준)|이용\s*(?:횟수|기간))(?:은|는|이|가|인가)?$')]
def classify(q):
    t=norm(q).rstrip(' ?.!,。？！')
    if any(p.search(t) for p in PROPER): return 'proper_noun'
    if '연회비 면제 조건' not in t and any(p.search(t) for p in NUMERIC): return 'numeric_condition'
    return 'semantic'
old_docs={c:r['document'] for c,r in old_by.items()}; new_docs={c:r['retrieval_text'] for c,r in new_by.items()}; pools={}; rrf_scores={}; routes={}
for q,text in query.items():
    routes[q]=classify(text)
    for name,docs,vecs,by in [('OLD',old_docs,ovec,old_by),('STRUCT',new_docs,nvec,new_by)]:
        vr=vector(qvec[q],vecs); br=bm25(text,docs); fused,score=rrf(vr,br); eligible=[c for c in fused if name=='STRUCT' or by[c]['metadata']['level'] in {'section','benefit'}]; require(len(eligible)>=20,f'{name}/{q} has fewer than 20 eligible chunks'); pools[(name,q)]=eligible[:20]; rrf_scores[(name,q)]={c:score[c] for c in eligible[:20]}
heading_re=re.compile(r'^\s{0,3}#{1,6}\s+(.+?)\s*$',re.MULTILINE)
def old_aug(cid):
    c=old_by[cid]; m=heading_re.search(c['document']); heading=m.group(1).strip() if m else ''; meta=c['metadata']; path=f"{meta['issuer']} > {meta['card_name']} > {meta['level']}"+(f' > {heading}' if heading else ''); return f'[문서 경로]\n{path}\n\n[본문]\n{c["document"]}'
bundles=[]
for q in query:
    by_card=defaultdict(list)
    for rank,cid in enumerate(pools[('STRUCT',q)],1): by_card[new_by[cid]['metadata']['card_key']].append((rank,cid))
    for card,seeds in sorted(by_card.items(),key=lambda x:(x[1][0][0],x[0])):
        best_rank,seed_id=seeds[0]; seed=new_by[seed_id]; node=node_by[seed['metadata']['node_id']]; selected=[]; trace=[]; parent_context=None
        def add(cid,kind,source_node):
            if len(selected)<5 and cid not in selected:
                require(new_by[cid]['metadata']['card_key']==card,'cross-card bundle'); selected.append(cid); trace.append({'order':len(selected),'chunk_id':cid,'relation':kind,'source_node_id':source_node})
        add(seed_id,'best_rrf_seed',node['node_id']); same=[new_by[c] for c in node['search_chunk_ids'] if c!=seed_id]; same.sort(key=lambda c:(abs(c['metadata']['part_index']-seed['metadata']['part_index']),c['metadata']['part_index'],c['chunk_id']))
        for c in same: add(c['chunk_id'],'same_node_adjacent_part',node['node_id'])
        parent=node_by.get(node['parent_id'])
        if parent is not None and parent['parent_id'] is not None:
            if not parent['heading_only']:
                for cid in parent['search_chunk_ids']: add(cid,'non_root_immediate_parent_direct_body',parent['node_id'])
            else:
                parent_context=parent['heading_text']; sl=node['heading_line_number'] if node['heading_line_number'] is not None else 10**12; siblings=[n for n in children[parent['node_id']] if not n['heading_only'] and n['search_chunk_ids']]; siblings.sort(key=lambda n:(abs((n['heading_line_number'] if n['heading_line_number'] is not None else 10**12)-sl),(n['heading_line_number'] if n['heading_line_number'] is not None else 10**12),n['node_id']))
                for n in siblings:
                    for cid in n['search_chunk_ids']: add(cid,'heading_only_parent_same_parent_direct_body_child',parent['node_id'])
        if node['parent_id'] is not None:
            for child in children[node['node_id']]:
                if not child['heading_only']:
                    for cid in child['search_chunk_ids']: add(cid,'seed_node_direct_child',node['node_id'])
        for _,cid in seeds[1:]: add(cid,'same_card_other_top20_seed',node['node_id'])
        sections=['[카드]\n'+seed['metadata']['issuer']+' > '+seed['metadata']['card_name']]
        if parent_context: sections.append('[상위 제목]\n'+parent_context)
        for i,cid in enumerate(selected,1):
            c=new_by[cid]; heading=' > '.join(c['heading_path']) if c['heading_path'] else '(root content)'; sections.append(f'[근거 {i} 경로]\n{heading}\n[근거 {i} 본문]\n{c["evidence_text"]}')
        text='\n\n'.join(sections); bundles.append({'query_id':q,'card_key':card,'best_seed_rrf_rank':best_rank,'best_seed_chunk_id':seed_id,'selected_chunk_ids':selected,'optional_parent_heading':parent_context,'bundle_text':text,'bundle_sha256':hashlib.sha256(text.encode()).hexdigest(),'relation_trace':trace,'root_children_fanout':False})
require(all(1<=len(b['selected_chunk_ids'])<=5 and not b['root_children_fanout'] for b in bundles),'bundle contract failed')
write_jsonl('label_free_bundles.jsonl',bundles)
pair_plan={}
for q in query:
    if routes[q]=='semantic':
        for cid in pools[('OLD',q)]: pair_plan[('OLD',q,cid)]={'text':old_aug(cid),'card_key':old_by[cid]['metadata']['card_key'],'rrf_rank':pools[('OLD',q)].index(cid)+1}
for b in bundles: pair_plan[('STRUCT',b['query_id'],b['bundle_sha256'])]={'text':b['bundle_text'],'card_key':b['card_key'],'rrf_rank':b['best_seed_rrf_rank']}
import torch
from transformers import AutoModelForSequenceClassification,AutoTokenizer
require(torch.cuda.is_available() and torch.cuda.device_count()==1 and '3090' in torch.cuda.get_device_name(0),'physical GPU0 RTX3090 unavailable')
tokenizer=AutoTokenizer.from_pretrained(BGE,local_files_only=True,trust_remote_code=False); require(tokenizer.model_max_length==8192,'tokenizer max mismatch')
audit=[]
for key,item in pair_plan.items():
    q=query[key[1]]; qn=len(tokenizer.encode(q,add_special_tokens=False)); dn=len(tokenizer.encode(item['text'],add_special_tokens=False)); total=len(tokenizer(q,item['text'],truncation='only_second',max_length=8192)['input_ids']); truncated=max(0,qn+dn+tokenizer.num_special_tokens_to_add(pair=True)-8192); require(truncated==0,'BGE truncation forbidden'); require(key[0]!='STRUCT' or dn<=4096,'structural bundle exceeds 4096 tokens'); audit.append({'package':key[0],'query_id':key[1],'candidate_id':key[2],'query_sha256':hashlib.sha256(q.encode()).hexdigest(),'input_sha256':hashlib.sha256(item['text'].encode()).hexdigest(),'input_tokens':total,'document_tokens':dn,'truncated_tokens':truncated,'gold_fields_used':False})
gpu_before=subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip(); started=time.perf_counter(); model=AutoModelForSequenceClassification.from_pretrained(BGE,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda:0'); load_seconds=time.perf_counter()-started; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); pairs=sorted(pair_plan); scores={}; started=time.perf_counter()
try:
    for i in range(0,len(pairs),2):
        part=pairs[i:i+2]; batch=tokenizer([query[k[1]] for k in part],[pair_plan[k]['text'] for k in part],padding=True,truncation='only_second',max_length=8192,return_tensors='pt').to('cuda:0')
        with torch.inference_mode(): values=model(**batch).logits.reshape(-1).float().cpu().numpy()
        require(len(values)==len(part) and np.isfinite(values).all(),'nonfinite BGE score'); scores.update({k:float(v) for k,v in zip(part,values)}); del batch,values
    torch.cuda.synchronize()
except torch.cuda.OutOfMemoryError:
    write_json('label_free_resources.json',{'status':'OOM_INVALID_NO_RETRY','batch_size':2,'partial_scores_discarded':True}); scores.clear(); torch.cuda.empty_cache(); raise RuntimeError('BGE OOM; partial scores discarded and no policy change permitted')
score_seconds=time.perf_counter()-started; peak_alloc=torch.cuda.max_memory_allocated(); peak_reserved=torch.cuda.max_memory_reserved(); del model,tokenizer; gc.collect(); torch.cuda.empty_cache(); gpu_after=subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],check=True,capture_output=True,text=True).stdout.strip(); require(set(scores)==set(pair_plan),'pair coverage mismatch')
with (OUT/'label_free_pair_scores.csv').open('w',encoding='utf-8',newline='') as f:
    fields=['package','query_id','candidate_id','card_key','original_rrf_rank','raw_logit']; w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows([{'package':k[0],'query_id':k[1],'candidate_id':k[2],'card_key':pair_plan[k]['card_key'],'original_rrf_rank':pair_plan[k]['rrf_rank'],'raw_logit':scores[k]} for k in pairs])
with (OUT/'label_free_scorer_input_audit.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(audit[0])); w.writeheader(); w.writerows(audit)
bundle_by={(b['query_id'],b['card_key']):b for b in bundles}; rankings=[]; payloads=[]
for q in query:
    base=pools[('OLD',q)]; ranked=sorted(base,key=lambda c:(-scores[('OLD',q,c)],base.index(c)+1,c)) if routes[q]=='semantic' else base; cards=[]
    for cid in ranked:
        card=old_by[cid]['metadata']['card_key']
        if card not in cards: cards.append(card)
    top=cards[:3]; rankings.append({'package':'OLD-D20-K3','query_id':q,'route':'reranker' if routes[q]=='semantic' else 'no_reranker','candidate_chunk_ids':base,'ranked_chunk_ids':ranked,'top3_card_keys':top})
    groups=[]
    for card in top:
        ids=[c for c in ranked if old_by[c]['metadata']['card_key']==card][:5]; groups.append({'card_key':card,'representative_id':ids[0],'source_ids':ids,'actual_body_texts':[old_by[c]['document'] for c in ids]})
    payloads.append({'package':'OLD-D20-K3','query_id':q,'groups':groups})
    bs=[b for b in bundles if b['query_id']==q]; bs.sort(key=lambda b:(-scores[('STRUCT',q,b['bundle_sha256'])],b['best_seed_rrf_rank'],b['card_key'])); topb=bs[:3]; rankings.append({'package':'STRUCT-D20-K3','query_id':q,'route':'reranker','candidate_chunk_ids':pools[('STRUCT',q)],'ranked_card_keys':[b['card_key'] for b in bs],'top3_card_keys':[b['card_key'] for b in topb]})
    payloads.append({'package':'STRUCT-D20-K3','query_id':q,'groups':[{'card_key':b['card_key'],'representative_id':b['best_seed_chunk_id'],'source_ids':b['selected_chunk_ids'],'actual_body_texts':[new_by[c]['evidence_text'] for c in b['selected_chunk_ids']],'bundle_sha256':b['bundle_sha256']} for b in topb]})
write_jsonl('label_free_rankings.jsonl',rankings); write_jsonl('label_free_payloads.jsonl',payloads)
write_json('label_free_resources.json',{'status':'PASS','physical_gpu':0,'model_revision':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e','batch_size':2,'dtype':'float16','max_length':8192,'truncation':'only_second','pair_count':len(scores),'old_pairs':sum(k[0]=='OLD' for k in scores),'struct_pairs':sum(k[0]=='STRUCT' for k in scores),'finite_pairs':len(scores),'truncated_pairs':0,'load_seconds':load_seconds,'scoring_seconds':score_seconds,'pairs_per_second':len(scores)/score_seconds,'peak_allocated_bytes':peak_alloc,'peak_reserved_bytes':peak_reserved,'gpu_before':gpu_before,'gpu_after':gpu_after,'api_requests':0,'network_requests':0,'new_embeddings':0,'chroma_queries':0})
freeze_files=['label_free_bundles.jsonl','label_free_pair_scores.csv','label_free_scorer_input_audit.csv','label_free_rankings.jsonl','label_free_payloads.jsonl','label_free_resources.json']
freeze_core={'status':'PASS_BEFORE_LABEL_LOAD','created_before_label_path_content_or_hash_access':True,'candidate_seal_sha256':seal['seal_core_sha256'],'query_cache_sha256':sha(cache),'files':{n:sha(OUT/n) for n in freeze_files},'execution_code_sha256':code_sha,'evaluation_code_sha256':cell_sha('26-label-evaluation'),'source_hashes':seal['seal_core']['source_hashes'],'model_hashes':{r:h for r,h in seal['seal_core']['source_hashes'].items() if r.startswith('.cache/reranker/bge-reranker-v2-m3/')},'ranking_rows':60,'payload_rows':60,'query_count':30,'labels_accessed':False}
write_json('label_free_execution_seal_v2.json',{'freeze_core':freeze_core,'freeze_core_sha256':hashlib.sha256(canon(freeze_core).encode()).hexdigest()})
for rel,digest in seal['seal_core']['source_hashes'].items(): require(sha(ROOT/rel)==digest,f'source changed after freeze: {rel}')
print({'label_free_freeze':'PASS','pairs':len(scores),'rankings':60,'labels_accessed':False})

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'label_free_freeze': 'PASS', 'pairs': 567, 'rankings': 60, 'labels_accessed': False}


In [ ]:
# Labels are first opened here, only after the label-free execution seal validates. CPU only.
from pathlib import Path
from collections import Counter
import csv,hashlib,json,math,os,statistics,unicodedata
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None
if ROOT is None: raise RuntimeError(f'Unexpected cwd: {cwd}')
NOTEBOOK=ROOT/'notebooks/26_cross_package_holdout_evaluation.ipynb'; OUT=ROOT/'notebooks/data/26_cross_package_holdout_evaluation'; H14=ROOT/'notebooks/data/14_retrieval_holdout'
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def require(ok,msg):
    if not ok: raise RuntimeError(msg)
def cell_sha(cid):
    d=json.loads(NOTEBOOK.read_text()); c=next((x for x in d['cells'] if x.get('id')==cid),None); require(c is not None,f'missing cell {cid}'); return hashlib.sha256(canon({'cell_id':cid,'source_text':''.join(c.get('source',[]))}).encode()).hexdigest()
def write_json(name,obj): (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def read_jsonl(name): return [json.loads(x) for x in (OUT/name).read_text().splitlines() if x.strip()]
require(os.environ.get('RUN_APPROVED_26_LABEL_EVAL')=='1','label evaluation not approved')
require(os.environ.get('RUN_APPROVED_26_QUERY_EMBEDDING','0')=='0' and os.environ.get('RUN_APPROVED_26_LABEL_FREE','0')=='0','external/GPU stages must be disabled during label evaluation')
eval_code_sha=cell_sha('26-label-evaluation'); require(os.environ.get('APPROVED_26_LABEL_EVAL_CODE_SHA256')==eval_code_sha,'evaluation code SHA mismatch')
freeze=json.loads((OUT/'label_free_execution_seal_v2.json').read_text()); core=freeze['freeze_core']; require(hashlib.sha256(canon(core).encode()).hexdigest()==freeze['freeze_core_sha256'],'execution seal mismatch')
require(core['status']=='PASS_BEFORE_LABEL_LOAD' and core['created_before_label_path_content_or_hash_access'] and core['evaluation_code_sha256']==eval_code_sha,'label-free sequence/code mismatch')
for name,digest in core['files'].items(): require(sha(OUT/name)==digest,f'frozen result changed: {name}')
for rel,digest in core['source_hashes'].items(): require(sha(ROOT/rel)==digest,f'source changed before labels: {rel}')
manifest=json.loads((H14/'holdout_data_manifest.json').read_text()); labels_path=H14/'holdout_labels.json'; require(sha(labels_path)==manifest['file_sha256']['notebooks/data/14_retrieval_holdout/holdout_labels.json'],'sealed labels mismatch')
labels=json.loads(labels_path.read_text())['labels']; queries=json.loads((H14/'holdout_queries.json').read_text())['queries']; require(len(labels)==len(queries)==30,'denominator mismatch'); label_by={r['query_id']:r for r in labels}; query_by={r['query_id']:r for r in queries}; require(set(label_by)==set(query_by),'query/label IDs mismatch')
old=[json.loads(x) for x in (ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').read_text().splitlines() if x.strip()]; new=[json.loads(x) for x in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl').read_text().splitlines() if x.strip()]
def norm(v): return ' '.join(unicodedata.normalize('NFKC',str(v)).casefold().split())
projection=[]; spans_by={}
for qid,label in label_by.items():
    seen=set(); spans=[]
    for e in label['relevant_chunks']:
        s=norm(e['evidence_span'])
        if s and s not in seen: spans.append(s); seen.add(s)
    require(bool(spans),f'empty frozen span set: {qid}'); spans_by[qid]=spans
    for i,s in enumerate(spans,1):
        old_ids=[r['id'] for r in old if s in norm(r['document'])]; new_ids=[r['chunk_id'] for r in new if s in norm(r['evidence_text'])]; projection.append({'query_id':qid,'span_index':i,'span_sha256':hashlib.sha256(s.encode()).hexdigest(),'old_projection_count':len(old_ids),'struct_projection_count':len(new_ids),'old_chunk_ids':old_ids,'struct_chunk_ids':new_ids})
missing=[r for r in projection if r['old_projection_count']==0 or r['struct_projection_count']==0]
if missing:
    write_json('decision.json',{'valid':False,'decision':'invalid','reason':'frozen evidence span does not project into both full corpora','missing_projection_units':[{'query_id':r['query_id'],'span_sha256':r['span_sha256'],'old':r['old_projection_count'],'struct':r['struct_projection_count']} for r in missing],'no_rerun_or_tuning_authorized':True}); raise RuntimeError('cross-corpus evidence projection invalid')
with (OUT/'evidence_projection.csv').open('w',encoding='utf-8',newline='') as f:
    fields=list(projection[0]); w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(projection)
rankings={(r['package'],r['query_id']):r for r in read_jsonl('label_free_rankings.jsonl')}; payloads={(r['package'],r['query_id']):r for r in read_jsonl('label_free_payloads.jsonl')}; require(len(rankings)==len(payloads)==60,'frozen row count mismatch')
per=[]
for package in ('OLD-D20-K3','STRUCT-D20-K3'):
    for qid in sorted(query_by):
        q=query_by[qid]; label=label_by[qid]; groups=payloads[(package,qid)]['groups']; cards=[g['card_key'] for g in groups]; require(cards==rankings[(package,qid)]['top3_card_keys'] and len(cards)==len(set(cards))<=3,'payload/ranking mismatch'); expected=label['expected_card']; card_rank=next((i for i,c in enumerate(cards,1) if c==expected),0); spans=spans_by[qid]; covered=set(); supported_rank=0
        for rank,g in enumerate(groups,1):
            actual=' '.join(norm(x) for x in g['actual_body_texts']); hits={i for i,s in enumerate(spans) if s in actual}; covered.update(hits)
            if g['card_key']==expected and hits and supported_rank==0: supported_rank=rank
        per.append({'package':package,'query_id':qid,'category':q['category'],'question_group':q['question_group'],'card_hit_at_3':int(card_rank>0),'card_mrr_at_3':1/card_rank if card_rank else 0.0,'supported_card_hit_at_3':int(q['question_group']=='evidence' and supported_rank>0),'supported_card_mrr_at_3':1/supported_rank if q['question_group']=='evidence' and supported_rank else 0.0,'evidence_span_coverage_at_3':len(covered)/len(spans) if q['question_group']=='evidence' else None,'complete_span_coverage_at_3':int(q['question_group']=='evidence' and len(covered)==len(spans))})
require(len(per)==60 and Counter(r['package'] for r in per)=={'OLD-D20-K3':30,'STRUCT-D20-K3':30},'per-query count mismatch')
with (OUT/'per_query_metrics.csv').open('w',encoding='utf-8',newline='') as f:
    fields=list(per[0]); w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(per)
groups={'all':lambda r:True,'card10':lambda r:r['question_group']=='card','evidence20':lambda r:r['question_group']=='evidence','numeric10':lambda r:r['category']=='numeric_condition','semantic10':lambda r:r['category']=='semantic_multi_condition'}; summary=[]
for package in ('OLD-D20-K3','STRUCT-D20-K3'):
    rows=[r for r in per if r['package']==package]
    for name,pred in groups.items():
        xs=[r for r in rows if pred(r)]; ev=[r for r in xs if r['question_group']=='evidence']; summary.append({'package':package,'group':name,'query_count':len(xs),'card_hit_at_3':sum(r['card_hit_at_3'] for r in xs)/len(xs),'card_hit_count':sum(r['card_hit_at_3'] for r in xs),'card_mrr_at_3':statistics.fmean(r['card_mrr_at_3'] for r in xs),'supported_card_hit_at_3':statistics.fmean(r['supported_card_hit_at_3'] for r in ev) if ev else None,'supported_hit_count':sum(r['supported_card_hit_at_3'] for r in ev),'supported_card_mrr_at_3':statistics.fmean(r['supported_card_mrr_at_3'] for r in ev) if ev else None,'evidence_span_coverage_at_3':statistics.fmean(r['evidence_span_coverage_at_3'] for r in ev) if ev else None,'complete_coverage_count':sum(r['complete_span_coverage_at_3'] for r in ev)})
with (OUT/'summary.csv').open('w',encoding='utf-8',newline='') as f:
    fields=list(summary[0]); w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(summary)
lookup={(r['package'],r['group']):r for r in summary}; paired=[]; wlt=[]
for metric in ('card_hit_at_3','card_mrr_at_3','supported_card_hit_at_3','supported_card_mrr_at_3','evidence_span_coverage_at_3'):
    domain='all' if metric.startswith('card_') else 'evidence20'; ids=[q for q in sorted(query_by) if domain=='all' or query_by[q]['question_group']=='evidence']; delta=[]
    for q in ids:
        a=next(r for r in per if r['package']=='OLD-D20-K3' and r['query_id']==q)[metric]; b=next(r for r in per if r['package']=='STRUCT-D20-K3' and r['query_id']==q)[metric]; delta.append((q,b-a)); paired.append({'metric':metric,'query_id':q,'old':a,'struct':b,'delta_struct_minus_old':b-a})
    w=sum(d>1e-12 for _,d in delta); l=sum(d<-1e-12 for _,d in delta); wlt.append({'metric':metric,'denominator':len(delta),'struct_wins':w,'struct_losses':l,'ties':len(delta)-w-l})
for name,rows in [('paired_deltas.csv',paired),('paired_wlt.csv',wlt)]:
    with (OUT/name).open('w',encoding='utf-8',newline='') as f: w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
absolute={}
for package in ('OLD-D20-K3','STRUCT-D20-K3'):
    a=lookup[(package,'all')]; e=lookup[(package,'evidence20')]; absolute[package]={'card_hit':a['card_hit_count']>=27,'supported_hit':e['supported_hit_count']>=14,'coverage':e['evidence_span_coverage_at_3']>=.65}; absolute[package]['pass']=all(absolute[package].values())
op='OLD-D20-K3'; sp='STRUCT-D20-K3'; oe=lookup[(op,'evidence20')]; se=lookup[(sp,'evidence20')]; oa=lookup[(op,'all')]; sa=lookup[(sp,'all')]; decision='inconclusive'
if absolute[op]['pass']!=absolute[sp]['pass']: decision=op if absolute[op]['pass'] else sp
elif not absolute[op]['pass']: decision='holdout_failed'
else:
    gap=se['supported_hit_count']-oe['supported_hit_count']
    if abs(gap)>=2:
        winner=sp if gap>0 else op; loser=op if gap>0 else sp; we=lookup[(winner,'evidence20')]; le=lookup[(loser,'evidence20')]; wa=lookup[(winner,'all')]; la=lookup[(loser,'all')]; decision=winner if wa['card_hit_at_3']+1e-12>=la['card_hit_at_3'] and we['evidence_span_coverage_at_3']+1e-12>=le['evidence_span_coverage_at_3'] and we['supported_card_mrr_at_3']+1e-12>=le['supported_card_mrr_at_3'] else 'inconclusive'
    elif gap==0 and abs(se['evidence_span_coverage_at_3']-oe['evidence_span_coverage_at_3'])>=.025:
        winner=sp if se['evidence_span_coverage_at_3']>oe['evidence_span_coverage_at_3'] else op; loser=op if winner==sp else sp; we=lookup[(winner,'evidence20')]; le=lookup[(loser,'evidence20')]; wa=lookup[(winner,'all')]; la=lookup[(loser,'all')]; coverage_wlt=next(r for r in wlt if r['metric']=='evidence_span_coverage_at_3'); wins=coverage_wlt['struct_wins'] if winner==sp else coverage_wlt['struct_losses']; losses=coverage_wlt['struct_losses'] if winner==sp else coverage_wlt['struct_wins']; decision=winner if we['supported_card_mrr_at_3']+1e-12>=le['supported_card_mrr_at_3'] and wa['card_hit_at_3']+1e-12>=la['card_hit_at_3'] and wins>losses else 'inconclusive'
decision_doc={'valid':True,'absolute_gates':absolute,'decision':decision,'winner_rule_applied_exact':True,'v2_not_v1_compliant_result':True,'low_score_rerun_forbidden':True,'holdout_text_in_summary':False}; write_json('decision.json',decision_doc); write_json('summary.json',{'summary':summary,'decision':decision_doc})
input_exact=all(sha(ROOT/rel)==digest for rel,digest in core['source_hashes'].items()); require(input_exact,'source changed after evaluation')
output_names=['evidence_projection.csv','per_query_metrics.csv','summary.csv','summary.json','paired_deltas.csv','paired_wlt.csv','decision.json']; write_json('resources.json',{'query_embedding_usage':json.loads((OUT/'query_embedding_usage.json').read_text()),'label_free_resources':json.loads((OUT/'label_free_resources.json').read_text()),'evaluation_cpu_only':True,'external_api_requests_during_evaluation':0,'network_during_evaluation':0,'gpu_during_evaluation':0})
output_names.append('resources.json'); write_json('integrity_final.json',{'status':'PASS_VALID_EVALUATION','label_free_seal_sha256':freeze['freeze_core_sha256'],'labels_sha256':sha(labels_path),'denominators':{'all':30,'card':10,'evidence':20,'numeric':10,'semantic':10},'projection_units':len(projection),'projection_missing':0,'ranking_rows':60,'payload_rows':60,'metric_ranges_pass':all((v is None or 0<=v<=1) for r in per for k,v in r.items() if k.endswith('_at_3')),'input_source_before_after_exact':input_exact,'api_network_gpu_model_chroma_during_evaluation':0,'output_hashes':{n:sha(OUT/n) for n in output_names}})
manifest_names=['candidate_execution_seal_v2.json','query_embedding_approval_manifest.json','query_embedding_attempt_ledger.json','query_embedding_usage.json','label_free_execution_seal_v2.json','label_free_bundles.jsonl','label_free_pair_scores.csv','label_free_scorer_input_audit.csv','label_free_rankings.jsonl','label_free_payloads.jsonl','label_free_resources.json']+output_names+['integrity_final.json']; write_json('run_manifest_final.json',{'manifest_self_hash_excluded':True,'files':{n:sha(OUT/n) for n in manifest_names},'notebook_code_cell_hashes':{cid:cell_sha(cid) for cid in ['26-approved-query-embedding','26-label-free-retrieval-bge','26-label-evaluation']},'v1_modified':False})
print({'evaluation':'PASS','decision':decision,'projection_units':len(projection),'holdout_text_reported':False})

## Terminal finalization — invalid cross-corpus evidence projection

사전 고정 validity gate가 구조 corpus에서 19개 frozen span unit을 exact projection하지 못해 의도대로 fail-closed했습니다. 이 결과는 완화 normalization, 인위적 제목 포함, 재검색, 재채점 또는 저점 재실행으로 바꾸지 않습니다. 아래 CPU/offline 셀은 기존 frozen 결과와 invalid decision을 byte-exact 검증한 뒤 집계 증거만 저장합니다.

In [1]:
# CPU/offline terminal record only: no retrieval, BGE, model, GPU, API, or network work.
from pathlib import Path
from collections import Counter
import csv,hashlib,json,os,unicodedata
cwd=Path.cwd().resolve(); ROOT=cwd if (cwd/'notebooks').is_dir() else cwd.parent if cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir() else None
if ROOT is None: raise RuntimeError(f'Unexpected cwd: {cwd}')
NOTEBOOK=ROOT/'notebooks/26_cross_package_holdout_evaluation.ipynb'; OUT=ROOT/'notebooks/data/26_cross_package_holdout_evaluation'; H14=ROOT/'notebooks/data/14_retrieval_holdout'
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def require(ok,msg):
    if not ok: raise RuntimeError(msg)
def write_json(name,obj): (OUT/name).write_text(json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True)+'\n',encoding='utf-8')
FROZEN={
 'decision.json':'a903ce55b108c71acd0b776242e8a442c423b3dea80ddd9f595553008e7c9ae1',
 'label_free_bundles.jsonl':'ee2b548c4d5d664b4da561dd828f61c3d528bc8157a8368da53afd3d0216720a',
 'label_free_execution_seal_v2.json':'e4f8533c02a9358d40730078971cd954750a96c39d435cba3843924b4996a35b',
 'label_free_pair_scores.csv':'3dae3ff5a1b70dd9684ff8c77b2d0a597a76b3971e7409d0bbc4f82a254b4b30',
 'label_free_payloads.jsonl':'ead33b80d4431ad05f1f51630c8fd7aa1641669e213f3bc148de3ff21869911a',
 'label_free_rankings.jsonl':'3919036aa51a2c5d5d478e1e10e1de571ab5070d35e8d38aafc8d326c123a497',
 'label_free_resources.json':'9804a8997a449b8e35f5a5b06c596adf0cd26315790afdfa24a9a0a40cb42b6e',
 'label_free_scorer_input_audit.csv':'6fad6786ade3def418b15308324da4aa2e9d55f81682f862bca897d04f041f6e',
 'query_embedding_usage.json':'4104e67e3a003e22b0106767411f0340771527ad6f8983da3945471ba2a317e5',
 'query_embedding_attempt_ledger.json':'214214cb5080e8744cab78196638dab5337e0e634087164b488a8e65c0b0d984',
}
for name,digest in FROZEN.items(): require(sha(OUT/name)==digest,f'frozen result changed: {name}')
decision=json.loads((OUT/'decision.json').read_text()); require(decision['valid'] is False and decision['decision']=='invalid' and decision['no_rerun_or_tuning_authorized'] is True and len(decision['missing_projection_units'])==19,'terminal decision mismatch')
usage=json.loads((OUT/'query_embedding_usage.json').read_text()); ledger=json.loads((OUT/'query_embedding_attempt_ledger.json').read_text()); resources=json.loads((OUT/'label_free_resources.json').read_text())
require(usage['requests']==1 and usage['provider_input_tokens']==1202 and usage['items']==30 and usage['retry_count']==0,'query API usage mismatch')
require(ledger['status']=='completed' and ledger['requests_started']==1 and ledger['retry_count']==0,'attempt ledger mismatch')
require(resources['status']=='PASS' and resources['pair_count']==resources['finite_pairs']==567 and resources['truncated_pairs']==0 and resources['physical_gpu']==0,'label-free resource mismatch')
manifest=json.loads((H14/'holdout_data_manifest.json').read_text())
for rel,digest in manifest['file_sha256'].items(): require(sha(ROOT/rel)==digest,f'v1 source changed: {rel}')
freeze=json.loads((OUT/'label_free_execution_seal_v2.json').read_text()); freeze_core=freeze['freeze_core']; require(hashlib.sha256(canon(freeze_core).encode()).hexdigest()==freeze['freeze_core_sha256'],'label-free seal mismatch')
for rel,digest in freeze_core['source_hashes'].items(): require(sha(ROOT/rel)==digest,f'package source changed: {rel}')
queries=json.loads((H14/'holdout_queries.json').read_text())['queries']; labels=json.loads((H14/'holdout_labels.json').read_text())['labels']; query_by={r['query_id']:r for r in queries}; require(len(query_by)==len(labels)==30,'holdout denominator mismatch')
old=[json.loads(x) for x in (ROOT/'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').read_text().splitlines() if x.strip()]; struct=[json.loads(x) for x in (ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl').read_text().splitlines() if x.strip()]
def norm(v): return ' '.join(unicodedata.normalize('NFKC',str(v)).casefold().split())
audit=[]
for label in labels:
    spans=list(dict.fromkeys(norm(e['evidence_span']) for e in label['relevant_chunks'] if norm(e['evidence_span']))); require(len(spans)==1,f'unexpected frozen span unit count: {label["query_id"]}')
    for index,span in enumerate(spans,1):
        old_count=sum(span in norm(r['document']) for r in old); struct_count=sum(span in norm(r['evidence_text']) for r in struct); q=query_by[label['query_id']]; audit.append({'query_id':label['query_id'],'question_group':q['question_group'],'category':q['category'],'span_unit_index':index,'span_sha256':hashlib.sha256(span.encode()).hexdigest(),'old_projection_count':old_count,'struct_projection_count':struct_count,'missing_in_old':int(old_count==0),'missing_in_struct':int(struct_count==0),'missing_any':int(old_count==0 or struct_count==0)})
require(len(audit)==30,'projection unit denominator mismatch'); missing=[r for r in audit if r['missing_any']]; require(len(missing)==19 and all(r['old_projection_count']>0 and r['struct_projection_count']==0 for r in missing),'missing projection result mismatch')
decision_missing={(r['query_id'],r['span_sha256'],r['old'],r['struct']) for r in decision['missing_projection_units']}; recomputed_missing={(r['query_id'],r['span_sha256'],r['old_projection_count'],r['struct_projection_count']) for r in missing}; require(decision_missing==recomputed_missing,'decision/recomputed missing units differ')
with (OUT/'terminal_projection_audit.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(audit[0])); w.writeheader(); w.writerows(audit)
group_missing=Counter(r['question_group'] for r in missing); category_missing=Counter(r['category'] for r in missing)
audit_summary={'status':'INVALID_CROSS_CORPUS_EVIDENCE_PROJECTION','projection_units_total':30,'projection_units_missing':19,'projection_units_old_present':sum(r['old_projection_count']>0 for r in audit),'projection_units_struct_present':sum(r['struct_projection_count']>0 for r in audit),'old_projection_match_count_total':sum(r['old_projection_count'] for r in audit),'struct_projection_match_count_total':sum(r['struct_projection_count'] for r in audit),'missing_by_question_group':dict(sorted(group_missing.items())),'missing_by_category':dict(sorted(category_missing.items())),'root_cause_categories':{'card_alias_actual_body_only_contract':group_missing['card'],'evidence20_exact_contiguous_span_absent_from_structural_evidence_text':group_missing['evidence']},'normalization_reused_exact':'NFKC + casefold + whitespace collapse','normalization_or_title_relaxation_attempted':False,'evidence_span_plaintext_persisted':False}
require(audit_summary['projection_units_old_present']==30 and audit_summary['projection_units_struct_present']==11 and audit_summary['old_projection_match_count_total']==71 and audit_summary['struct_projection_match_count_total']==11,'projection aggregate mismatch')
require(audit_summary['missing_by_question_group']=={'card':10,'evidence':9} and audit_summary['missing_by_category']=={'card_alias':10,'numeric_condition':4,'semantic_multi_condition':5},'missing breakdown mismatch')
write_json('terminal_projection_audit_summary.json',audit_summary)
terminal={'status':'INVALID_CROSS_CORPUS_EVIDENCE_PROJECTION','terminal':True,'winner':None,'intentional_fail_closed':True,'predeclared_validity_gate_unchanged':True,'ranking_bundle_pair_resources_decision_byte_exact':True,'api_usage':{'requests':1,'items':30,'tokens':1202,'retry_count':0,'actual_cost_usd':usage['actual_cost_usd']},'label_free':{'pair_count':567,'old_pairs':resources['old_pairs'],'struct_pairs':resources['struct_pairs'],'finite_pairs':567,'truncated_pairs':0,'model_revision':resources['model_revision'],'batch_size':resources['batch_size'],'dtype':resources['dtype'],'load_seconds':resources['load_seconds'],'scoring_seconds':resources['scoring_seconds'],'peak_allocated_bytes':resources['peak_allocated_bytes'],'peak_reserved_bytes':resources['peak_reserved_bytes']},'current_finalization':{'api':0,'network':0,'new_embedding':0,'gpu':0,'model_load':0,'retrieval':0,'bge_scoring':0,'chroma':0},'no_holdout_reuse_or_tuning':True,'required_next_step':'new structure-neutral re-annotated holdout declared and sealed before results','v1_modified':False}
write_json('terminal_finalization.json',terminal)
marker='## Terminal v2 disposition'
readme=(OUT/'README.md').read_text()
if marker not in readme: (OUT/'README.md').write_text(readme+'\n'+marker+'\n\n`INVALID_CROSS_CORPUS_EVIDENCE_PROJECTION`으로 종료했습니다. 전체 30 span unit 중 19개가 structural actual evidence body에 exact projection되지 않았습니다. 기존 ranking/BGE 결과와 invalid 판정은 재사용·완화·튜닝하지 않으며 winner는 없습니다. 카드/별칭 10건은 인위적 제목을 근거로 인정하지 않는 actual-body-only 계약의 영향을 받았고, evidence 20건 중 9건도 structural evidence text에 frozen contiguous span이 없었습니다. 평문 span은 이 보고서에 남기지 않았습니다. 다음 비교에는 결과를 보기 전에 새 structure-neutral annotation을 별도로 seal해야 합니다.\n',encoding='utf-8')
notebook=json.loads(NOTEBOOK.read_text()); notebook_source_sha=hashlib.sha256(canon([{'id':c.get('id'),'cell_type':c['cell_type'],'source_text':''.join(c.get('source',[]))} for c in notebook['cells']]).encode()).hexdigest()
new_files=['terminal_projection_audit.csv','terminal_projection_audit_summary.json','terminal_finalization.json','README.md']; integrity={'status':'PASS_TERMINAL_INVALID_FINALIZATION','terminal_status':terminal['status'],'winner':None,'saved_execution_error_outputs_expected':0,'projection_units':30,'missing_units':19,'frozen_files_sha256':FROZEN,'frozen_files_byte_exact':all(sha(OUT/n)==d for n,d in FROZEN.items()),'v1_manifest_files_exact':True,'package_sources_exact':True,'notebook_source_sha256':notebook_source_sha,'current_run_api_network_gpu_model_retrieval_bge':0,'new_output_hashes':{n:sha(OUT/n) for n in new_files}}
write_json('integrity_final.json',integrity)
manifest_files=list(FROZEN)+new_files+['integrity_final.json']; write_json('run_manifest_final.json',{'manifest_self_hash_excluded':True,'terminal_status':terminal['status'],'files':{n:sha(OUT/n) for n in manifest_files},'notebook_source_sha256':notebook_source_sha,'v1_modified':False,'current_run_external_or_gpu_calls':0})
for name,digest in FROZEN.items(): require(sha(OUT/name)==digest,f'frozen result changed during finalization: {name}')
print({'terminal_status':terminal['status'],'projection_units':30,'missing':19,'missing_by_group':dict(group_missing),'missing_by_category':dict(category_missing),'winner':None,'current_api_gpu_model_retrieval_bge':0})

{'terminal_status': 'INVALID_CROSS_CORPUS_EVIDENCE_PROJECTION', 'projection_units': 30, 'missing': 19, 'missing_by_group': {'card': 10, 'evidence': 9}, 'missing_by_category': {'card_alias': 10, 'semantic_multi_condition': 5, 'numeric_condition': 4}, 'winner': None, 'current_api_gpu_model_retrieval_bge': 0}
